In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
# arguments
protein = "GPR37L1"

In [ ]:
# loading the data
residuals = f"/Volumes/Intenso/image-analysis/synapse-counting/results_202512/{protein}_lmem_residuals_results.csv"
residuals_df = pd.read_csv(residuals, index_col=0)
residuals_df.rename_axis("sample_ID")

In [ ]:
# scaling the data
scaler = StandardScaler()
scaled_residuals = scaler.fit_transform(residuals_df)

# put back into df
scaled_residuals_df = pd.DataFrame(data=scaled_residuals,
                     columns=residuals_df.columns,
                     index=residuals_df.index)
scaled_residuals_df = scaled_residuals_df.rename_axis("sample_ID")
scaled_residuals_df

In [ ]:
sample_hemispheres = [idx.split('_')[0] for idx in scaled_residuals_df.index]
sample_brain_ids = [idx.split('_')[1] for idx in scaled_residuals_df.index]

# create color mappings

# for Hemisphere
if protein == "VCAM1":
    hemisphere_color_map = {"LacZ-gRNA": "#c6c6c6", "VCAM1-gRNA": "#21a0e2"}
else:
    protein == "GPR37L1"
    hemisphere_color_map = {"LacZ-gRNA": "#c6c6c6", "GPR37L1-gRNA": "#e28d21"}

row_colors_hemisphere = [hemisphere_color_map[h] for h in sample_hemispheres]

# for BrainID (you have 'Brain-4', 'Brain-4-2', 'Brain-5', 'Brain-7')
unique_brain_ids = sorted(list(set(sample_brain_ids)))
brain_palette = sns.color_palette("Dark2", len(unique_brain_ids))
brain_color_map = {brain_id_str: brain_palette[i] for i, brain_id_str in enumerate(unique_brain_ids)}
row_colors_brain = [brain_color_map[b] for b in sample_brain_ids]

# combine into a DataFrame for clustermap's row_colors
row_colors_df = pd.DataFrame({
    'Hemisphere': row_colors_hemisphere,
    'BrainID': row_colors_brain
}, index=scaled_residuals_df.index)

In [ ]:
selected_features = {
    "residual_VGLUT1_PSD95_CA3_SL_local_peak_colocalized_spots", 
    "residual_VGLUT1_PSD95_CA3_SR_local_peak_colocalized_spots",
} 

In [ ]:
g = sns.clustermap(
        scaled_residuals_df,
        method="weighted",      
        metric="correlation",    
        cmap="vlag",        
        row_colors=row_colors_df, 
        col_cluster=True,  
        row_cluster=True,   
        dendrogram_ratio=(.05, .2),
        cbar_pos=(1, .8, .02, .18), 
        # annot = True,
        # annot_kws={"size": 5},
        figsize=(20, 10) 
)

plt.show()
plt.suptitle('Heatmap of Brain-Adjusted Features', y=1.02, fontsize=16) # Title for the figure
plt.show()

In [ ]:
g = sns.clustermap(
    scaled_residuals_df,
    method="weighted",
    metric="correlation",
    cmap="vlag",
    row_colors=row_colors_df,
    col_cluster=True,
    row_cluster=True,
    dendrogram_ratio=(.05, .2),
    cbar_pos=(1, .8, .02, .18),
    figsize=(20, 10)
)

clustered_cols = list(g.data2d.columns)

# positions of selected features in clustered order
sel_pos = [i for i, c in enumerate(clustered_cols) if c in selected_features]
sel_labels = [clustered_cols[i] for i in sel_pos]

ax = g.ax_heatmap

# set ONLY these ticks + labels
ax.set_xticks(sel_pos)
ax.set_xticklabels(sel_labels, rotation=90, fontsize=9)

# remove the little minor ticks that seaborn/matplotlib may add
ax.tick_params(axis="x", which="minor", bottom=False)
ax.tick_params(axis="x", which="major", bottom=True, length=4)

# give labels breathing room
g.fig.subplots_adjust(bottom=0.30)

# adjust dendrogram line thickness 
lw = 1.5 

for ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
    # seaborn draws dendrograms as LineCollections
    for coll in ax.collections:
        coll.set_linewidth(lw)
    # sometimes there are also plain Line2D objects
    for line in ax.lines:
        line.set_linewidth(lw)

plt.show()

plt.show()